## Импорты и параметры

### Привязываем Гугл-диск

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Импортируем библиотеки

In [ ]:
!pip install python-docx

import time
import random
import re
import pandas as pd
from docx import Document
import numpy as np
import os
import json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 107.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.3 MB/s eta 0:00:00
  Attempting uninstall: trl
    Found existing installation: trl 1.4.0
    Uninstalling trl-1.4.0:
      Successfully uninstalled trl-1.4.0
Found existing installation: pyarrow 24.0.0

229

### Сохраняем рандом сид

In [ ]:
def set_random_seed(seed: int = 27):
    random.seed(seed)
    np.random.seed(seed)

set_random_seed()

### Сохраняем важные параметры

In [ ]:
DATA_FILES = [
    ('pul_intervyu_1.docx', 'kodirovki_pfi_2023.docx'),
    ('pul_intervyu_2.docx', 'kodirovki_pfi_2024.docx'),
    ('pul_intervyu_3.docx', 'kodirovki_sp_2024.docx'),
    ('pul_intervyu_4.docx', 'kodirovki_pfi_2025.docx'),
]

## Создание датасета

### Загрузка и преобразование вордовских файлов

In [ ]:
def parse_interview_transcript(filename):
    """Парсинг транскриптов интервью"""
    try:
        doc = Document(filename)
    except Exception as e:
        print(f"Ошибка при открытии {filename}: {e}")
        return []

    paragraphs = [para.text.strip() for para in doc.paragraphs if para.text.strip()]

    if not paragraphs:
        return []

    global_topic = ""
    if not re.match(r'^Интервью\s+\d+', paragraphs[0]):
        global_topic = paragraphs[0]

    interviews = []
    current_interview = None

    for text in paragraphs:
        if text == global_topic:
            continue

        match = re.match(r'^Интервью\s+(\d+)', text)
        if match:
            if current_interview:
                interviews.append(current_interview)
            interview_num = match.group(1)
            current_interview = {
                'interview_num': interview_num,
                'topic': global_topic,
                'transcript': ''
            }
        elif current_interview:
            if current_interview['transcript']:
                current_interview['transcript'] += '\n' + text
            else:
                current_interview['transcript'] = text

    if current_interview:
        interviews.append(current_interview)

    return interviews


def parse_coding(filename):
    """Парсинг кодировок интервью"""
    try:
        doc = Document(filename)
    except Exception as e:
        print(f"Ошибка при открытии {filename}: {e}")
        return []

    paragraphs = [para.text.strip() for para in doc.paragraphs if para.text.strip()]

    interviews = []
    current_interview = None

    for text in paragraphs:
        match = re.match(r'^Интервью\s+(\d+)', text)
        if match:
            if current_interview:
                interviews.append(current_interview)
            interview_num = match.group(1)
            current_interview = {
                'interview_num': interview_num,
                'coding': ''
            }
        elif current_interview:
            if current_interview['coding']:
                current_interview['coding'] += '\n' + text
            else:
                current_interview['coding'] = text

    if current_interview:
        interviews.append(current_interview)

    return interviews


def load_all_data():
    """Загрузка и объединение всех данных"""
    all_interviews = []

    for trans_file, code_file in DATA_FILES:
        print(f"Обработка {trans_file} и {code_file}...")

        transcripts = parse_interview_transcript(trans_file)
        codings = parse_coding(code_file)

        if not transcripts:
            print(f"Предупреждение: Не удалось загрузить транскрипты из {trans_file}")
            continue

        coding_dict = {c['interview_num']: c['coding'] for c in codings}

        for trans in transcripts:
            interview_num = trans['interview_num']
            coding = coding_dict.get(interview_num, '')

            if coding:
                transcript = trans['transcript']

                all_interviews.append({
                    'id': len(all_interviews),
                    'topic': trans['topic'],
                    'interview_num': interview_num,
                    'transcript': transcript,
                    'coding': coding
                })

    df = pd.DataFrame(all_interviews)
    print(f"Всего загружено интервью с кодировками: {len(df)}")
    return df

In [ ]:
df_interviews = load_all_data()

Обработка pul_intervyu_1.docx и kodirovki_pfi_2023.docx...
Обработка pul_intervyu_2.docx и kodirovki_pfi_2024.docx...
Обработка pul_intervyu_3.docx и kodirovki_sp_2024.docx...
Обработка pul_intervyu_4.docx и kodirovki_pfi_2025.docx...
Всего загружено интервью с кодировками: 150


In [ ]:
df_interviews.head()

,id,topic,interview_num,transcript,coding
0,0,"Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...",1,"Интервьюер: Так, все, все записи начались. Сна...","**Общий код 1: Поколенческие характеристики, ц..."
1,1,"Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...",2,"Интервьюер: Смотри, вначале расскажи, пожалуйс...","**Общий код 1: Поколенческие характеристики, ц..."
2,2,"Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...",3,"Интервьюер: Так, тогда начинаем. Расскажи спер...","**Общий код 1: Поколенческие характеристики, ц..."
3,3,"Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...",4,"Интервьюер: Тогда, наверное, начнем. Сперва мо...","**Общий код 1: Поколенческие характеристики, ц..."
4,4,"Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...",5,Интервьюер: Для начала расскажи о себе. Скольк...,"**Общий код 1: Поколенческие характеристики, ц..."


In [ ]:
print(f'''Уникальных тем: {df_interviews['topic'].nunique()}
Уникальных текстов: {df_interviews['transcript'].nunique()}
Уникальных кодировок: {df_interviews['coding'].nunique()}''')

Уникальных тем: 4
Уникальных текстов: 150
Уникальных кодировок: 150


### Сплит и сохранение файлов

In [ ]:
def split_dataset(df, to_csv = False, path = None):
    """
    Разделение данных на train/val/test
    path -- папка (на гугл диске), куда сохранятся датасеты
    """
    indices = np.random.permutation(len(df))

    n_train = int(0.7 * len(df))
    n_val = int(0.15 * len(df))

    train_df = df.iloc[indices[:n_train]]
    val_df = df.iloc[indices[n_train:n_train + n_val]]
    test_df = df.iloc[indices[n_train + n_val:]]

    print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

    if to_csv:
        if path:
            train_df.to_csv(f'{path}/train_data.csv', encoding='utf-8')
            val_df.to_csv(f'{path}/val_data.csv', encoding='utf-8')
            test_df.to_csv(f'{path}/test_data.csv', encoding='utf-8')
        else:
            train_df.to_csv('train_data.csv', encoding='utf-8')
            val_df.to_csv('val_data.csv', encoding='utf-8')
            test_df.to_csv('test_data.csv', encoding='utf-8')

    return train_df, val_df, test_df

In [ ]:
train, val, test = split_dataset(df_interviews, to_csv=True)

Train: 105, Val: 22, Test: 23
